In [1]:
import pandas as pd
import numpy as np

dataset = pd.read_csv("house_prices.csv")
dataset.head(10)

,DienTich_m2,SoPhongNgu,KhoangCach_TT_km,TuoiNha_Nam,SoTang,GiaNha_TyVND
0,132.6,4,2.0,17,5,9.58
1,42.5,1,2.4,20,3,3.27
2,116.0,3,4.9,18,2,7.85
3,129.9,3,2.8,0,2,9.78
4,99.8,3,6.3,18,2,6.56
5,67.0,2,18.4,8,3,3.47
6,63.8,2,10.0,8,4,4.48
7,131.3,4,3.9,11,5,9.85
8,60.3,2,16.5,8,3,3.38
9,50.6,1,9.0,8,5,2.66


In [6]:
from sklearn.model_selection import train_test_split

feature_names = ['DienTich_m2', 'SoPhongNgu', 'KhoangCach_TT_km', 'TuoiNha_Nam',
       'SoTang']

X = dataset[feature_names]
y = dataset['GiaNha_TyVND'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Số lượng mẫu Train: {len(X_train)} | Số lượng mẫu Test: {len(X_test)}')


Số lượng mẫu Train: 80 | Số lượng mẫu Test: 20


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_train_pred = lin_reg.predict(X_train)
y_test_pred = lin_reg.predict(X_test)

print(f'Hệ số chặn (Intercept w0): {lin_reg.intercept_:,.2f}')
for name, coef in zip(feature_names, lin_reg.coef_):
    print(f'Trọng số {name} (w): {coef:.4f}')

print('-' * 40)
print(f'R² Score trên Train Set: {r2_score(y_train, y_train_pred):.4f}')
print(f'RMSE trên Train Set:     ${root_mean_squared_error(y_train, y_train_pred):,.2f}')

print(f'R² Score trên Test Set: {r2_score(y_test, y_test_pred):.4f}')
print(f'RMSE trên Test Set:     ${root_mean_squared_error(y_test, y_test_pred):,.2f}')


Hệ số chặn (Intercept w0): 1.64
Trọng số DienTich_m2 (w): 0.0578
Trọng số SoPhongNgu (w): 0.3260
Trọng số KhoangCach_TT_km (w): -0.1343
Trọng số TuoiNha_Nam (w): -0.0457
Trọng số SoTang (w): 0.0211
----------------------------------------
R² Score trên Train Set: 0.9727
RMSE trên Train Set:     $0.39
R² Score trên Test Set: 0.9771
RMSE trên Test Set:     $0.38


In [10]:
pip install statsmodels

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import numpy as np
import statsmodels.api as sm
# Thêm cột hệ số tự do w0 (toàn số 1: x0 = 1) vào ma trận X
X_opt = np.append(arr=np.ones((X.shape[0], 1)).astype(float), values=X, axis=1)
all_feature_names = ['Intercept (w0)'] + feature_names

# Huấn luyện mô hình OLS và truyền xname để hiển thị rõ tên từng đặc trưng
regressor_ols = sm.OLS(endog=y, exog=X_opt).fit()
print(regressor_ols.summary(xname=all_feature_names))


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.974
Model:                            OLS   Adj. R-squared:                  0.973
Method:                 Least Squares   F-statistic:                     714.5
Date:                Sun, 27 Sep 2026   Prob (F-statistic):           4.00e-73
Time:                        15:58:15   Log-Likelihood:                -46.321
No. Observations:                 100   AIC:                             104.6
Df Residuals:                      94   BIC:                             120.3
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept (w0)       1.5455      0.182  

In [22]:
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.linear_model import LinearRegression

# 1. Chọn 4 đặc trưng tối ưu (đã loại bỏ 'SoTang')
feature_names_opt = ['DienTich_m2', 'SoPhongNgu', 'KhoangCach_TT_km', 'TuoiNha_Nam']

# 2. Huấn luyện mô hình tối ưu mới với 4 đặc trưng
lin_reg_opt = LinearRegression()
lin_reg_opt.fit(X_train[feature_names_opt], y_train)

# 3. Chuẩn bị tập kiểm tra 4 đặc trưng
X_test_opt = X_test[feature_names_opt]
X_test_arr = np.array(X_test_opt)
y_test_arr = np.array(y_test)

# Dự đoán giá nhà trên tập test
y_pred = lin_reg_opt.predict(X_test_opt)

# ==============================================================================
# 1. Xây dựng hàm compare() so sánh từng căn nhà riêng lẻ (4 đặc trưng)
# ==============================================================================
def compare(i_example):
    # Lấy mẫu thứ i_example
    x = X_test_arr[i_example : i_example + 1]
    y_actual = y_test_arr[i_example]
    y_predicted = lin_reg_opt.predict(X_test_opt.iloc[i_example : i_example + 1])[0]
    
    diff = y_actual - y_predicted
    error_pct = abs(diff) / y_actual * 100

    print(
        f"Mẫu {i_example+1:2d} | "
        f"DT: {x[0, 0]:5.1f} m² | "
        f"PN: {int(x[0, 1]):1d} | "
        f"KC TT: {x[0, 2]:4.1f} km | "
        f"Tuổi: {int(x[0, 3]):2d} năm | "
        f"Thực tế: {y_actual:5.2f} tỷ | "
        f"Dự đoán: {y_predicted:5.2f} tỷ | "
        f"Lệch: {diff:+5.2f} tỷ ({error_pct:5.2f}%)"
    )

print("=== Thử nghiệm hàm compare() với căn nhà đầu tiên trong tập Test ===")
compare(0)


# ==============================================================================
# 2. Lập bảng tổng hợp so sánh toàn bộ các mẫu trong tập Test (Không còn cột Số tầng)
# ==============================================================================
df_compare = pd.DataFrame({
    'STT': range(1, len(X_test_arr) + 1),
    'Diện tích (m²)': [f"{v:.1f}" for v in X_test_arr[:, 0]],
    'Số PN': [int(v) for v in X_test_arr[:, 1]],
    'KC Trung tâm (km)': [f"{v:.1f}" for v in X_test_arr[:, 2]],
    'Tuổi nhà (năm)': [int(v) for v in X_test_arr[:, 3]],
    'Giá thực tế (Tỷ)': [f"{v:.2f}" for v in y_test_arr],
    'Giá dự đoán (Tỷ)': [f"{v:.2f}" for v in y_pred],
    'Chênh lệch (Tỷ)': [f"{v:+.2f}" for v in (y_test_arr - y_pred)],
    'Sai số (%)': [f"{abs(act - pred)/act * 100:.2f}%" for act, pred in zip(y_test_arr, y_pred)]
})

print("\n=== Bảng Đối chiếu Toàn bộ Tập Test (Mô hình tối ưu - 4 đặc trưng) ===")
display(df_compare)


=== Thử nghiệm hàm compare() với căn nhà đầu tiên trong tập Test ===
Mẫu  1 | DT:  71.5 m² | PN: 2 | KC TT:  7.0 km | Tuổi: 18 năm | Thực tế:  4.41 tỷ | Dự đoán:  4.73 tỷ | Lệch: -0.32 tỷ ( 7.17%)

=== Bảng Đối chiếu Toàn bộ Tập Test (Mô hình tối ưu - 4 đặc trưng) ===


,STT,Diện tích (m²),Số PN,KC Trung tâm (km),Tuổi nhà (năm),Giá thực tế (Tỷ),Giá dự đoán (Tỷ),Chênh lệch (Tỷ),Sai số (%)
0,1,71.5,2,7.0,18,4.41,4.73,-0.32,7.17%
1,2,58.1,1,16.8,0,2.48,3.13,-0.65,26.36%
2,3,44.8,2,18.1,8,2.65,2.16,+0.49,18.60%
3,4,130.6,4,3.8,7,9.71,9.73,-0.02,0.22%
4,5,82.2,2,6.4,12,5.59,5.70,-0.11,1.96%
5,6,115.2,4,6.8,6,8.88,8.49,+0.39,4.43%
6,7,84.8,3,11.0,6,6.16,5.84,+0.32,5.20%
7,8,112.7,3,12.6,16,6.41,6.77,-0.36,5.62%
8,9,122.2,4,2.0,16,9.43,9.07,+0.36,3.77%
9,10,132.6,4,2.0,17,9.58,9.63,-0.05,0.50%


In [23]:
from IPython.display import display
display(df_compare)

,STT,Diện tích (m²),Số PN,KC Trung tâm (km),Tuổi nhà (năm),Giá thực tế (Tỷ),Giá dự đoán (Tỷ),Chênh lệch (Tỷ),Sai số (%)
0,1,71.5,2,7.0,18,4.41,4.73,-0.32,7.17%
1,2,58.1,1,16.8,0,2.48,3.13,-0.65,26.36%
2,3,44.8,2,18.1,8,2.65,2.16,+0.49,18.60%
3,4,130.6,4,3.8,7,9.71,9.73,-0.02,0.22%
4,5,82.2,2,6.4,12,5.59,5.70,-0.11,1.96%
5,6,115.2,4,6.8,6,8.88,8.49,+0.39,4.43%
6,7,84.8,3,11.0,6,6.16,5.84,+0.32,5.20%
7,8,112.7,3,12.6,16,6.41,6.77,-0.36,5.62%
8,9,122.2,4,2.0,16,9.43,9.07,+0.36,3.77%
9,10,132.6,4,2.0,17,9.58,9.63,-0.05,0.50%


In [27]:
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression

# 1. Định nghĩa 4 đặc trưng tối ưu (đã bỏ SoTang)
feature_cols = ['DienTich_m2', 'SoPhongNgu', 'KhoangCach_TT_km', 'TuoiNha_Nam']

# 2. Đảm bảo mô hình tối ưu lin_reg_opt được huấn luyện với 4 đặc trưng
lin_reg_opt = LinearRegression()
lin_reg_opt.fit(X_train[feature_cols], y_train)

# 3. Định nghĩa các kịch bản theo yêu cầu đề bài
scenarios = pd.DataFrame([
    {
        'Kịch bản': '1. Giá thấp nhất (Min / Tiết kiệm)',
        'DienTich_m2': 120.0,
        'SoPhongNgu': 4,
        'KhoangCach_TT_km': 10.0,
        'TuoiNha_Nam': 5,
    },
    {
        'Kịch bản': '2. Giá trung bình (Điển hình)',
        'DienTich_m2': 135.0,
        'SoPhongNgu': 4,
        'KhoangCach_TT_km': 5.0,
        'TuoiNha_Nam': 2,
    },
    {
        'Kịch bản': '3. Giá cao nhất (Max / Cao cấp)',
        'DienTich_m2': 150.0,
        'SoPhongNgu': 4,
        'KhoangCach_TT_km': 2.0,
        'TuoiNha_Nam': 0,
    }
])

X_query = scenarios[feature_cols]

# 4. Dự đoán bằng mô hình tối ưu lin_reg_opt (KHÔNG dùng lin_reg cũ)
scenarios['Giá dự đoán (Tỷ VNĐ)'] = lin_reg_opt.predict(X_query)
scenarios['Giá quy đổi (Triệu VNĐ)'] = scenarios['Giá dự đoán (Tỷ VNĐ)'].apply(lambda x: f"{x * 1000:,.0f} triệu")
scenarios['Giá dự đoán (Tỷ VNĐ)'] = scenarios['Giá dự đoán (Tỷ VNĐ)'].apply(lambda x: f"{x:.2f} tỷ")

print("=== BẢNG DỰ ĐOÁN NGÂN SÁCH MUA NHÀ CHO KHÁCH HÀNG (MÔ HÌNH 4 ĐẶC TRƯNG) ===")
display(scenarios[['Kịch bản', 'DienTich_m2', 'SoPhongNgu', 'KhoangCach_TT_km', 'TuoiNha_Nam', 'Giá dự đoán (Tỷ VNĐ)', 'Giá quy đổi (Triệu VNĐ)']])


=== BẢNG DỰ ĐOÁN NGÂN SÁCH MUA NHÀ CHO KHÁCH HÀNG (MÔ HÌNH 4 ĐẶC TRƯNG) ===


,Kịch bản,DienTich_m2,SoPhongNgu,KhoangCach_TT_km,TuoiNha_Nam,Giá dự đoán (Tỷ VNĐ),Giá quy đổi (Triệu VNĐ)
0,1. Giá thấp nhất (Min / Tiết kiệm),120.0,4,10.0,5,8.38 tỷ,"8,379 triệu"
1,2. Giá trung bình (Điển hình),135.0,4,5.0,2,10.05 tỷ,"10,053 triệu"
2,3. Giá cao nhất (Max / Cao cấp),150.0,4,2.0,0,11.41 tỷ,"11,412 triệu"


Vậy vị khách này phải chi trong khoảng 8.38 tỷ VND đến 11.41 tỷ VND